# 中证800 V69 V46模型诊断：优势与失效归因

目标：冻结当前 V46-family 训练方式，把模型吃透。

本 notebook 不做策略优化，不新增因子，不做 ensemble，不改交易规则。它只回答：

1. V46 模型到底靠哪些特征/风格赚钱？
2. top8、top20、score bucket 的收益衰减是否合理？
3. 2022/2023 和典型失败月到底亏在哪里？
4. 问题更像来自训练样本、因子选择、label 噪声，还是 top8 集中度？

固定训练口径：full features + LGB direct + fixed120 + alpha_1m + legacy_rebalance boundary。

In [ ]:
import os
import gc
import json
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 260)
pd.set_option("display.width", 260)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", leave=True):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, leave=leave)
    def _gen():
        every = max(1, int((total or 100) / 20))
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                print("%s %s%s" % (desc, i, "/%s" % total if total else ""))
            yield item
    return _gen()


def display_df(df, n=30):
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=False))

## 1. 固定配置

只改 `DIAG_MODEL_SPECS` 可以选择要诊断哪些 V46-family cutoff。默认覆盖早期弱窗口和当前主线窗口。

In [ ]:
PROJECT_DIR = Path("/Users/youzou/Documents/New project/quant-research/机器学习策略")
OUT_DIR = PROJECT_DIR / "csi800_ml_v69_v46_model_diagnostics_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    Path("train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("train_csi800_factor_v40_data_enhancement.csv"),
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv",
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement.csv",
    Path.home() / "Downloads" / "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv",
    Path.home() / "Downloads" / "train_csi800_factor_v40_data_enhancement.csv",
]
DATA_PATH_OVERRIDE = None

TARGET_COL = "alpha_1m"
STOCK_COL = "stock"
DATE_COL = "rebalance_date"
INDUSTRY_COL = "industry_bucket"
BENCHMARK = "000906.XSHG"

FIXED_ITER = 120
SEED = 42
CORR_THRESHOLD = 0.70
TOP_N_CANDIDATES = 30
STOCK_NUM = 8
PORTFOLIO_RULE = "top8_board_cap"
BOARD_CAPS = {"chinext": 3, "star": 2}
BOARD_CAPS_TEXT = ";".join(["%s:%s" % (k, BOARD_CAPS[k]) for k in sorted(BOARD_CAPS)])
LABEL_BOUNDARY_MODE = "legacy_rebalance"

# Frozen V46-family diagnostics. Do not optimize here.
DIAG_MODEL_SPECS = [
    {"model_tag": "exp_2021_12", "train_start": "2019-01-01", "train_end": "2021-12-31", "test_start": "2022-01-01", "test_end": "2023-12-31", "note": "early 36m sample; diagnose 2022/2023"},
    {"model_tag": "exp_2022_12", "train_start": "2019-01-01", "train_end": "2022-12-31", "test_start": "2023-01-01", "test_end": "2024-12-31", "note": "48m sample; transition window"},
    {"model_tag": "exp_2023_12", "train_start": "2019-01-01", "train_end": "2023-12-31", "test_start": "2024-01-01", "test_end": "2025-12-31", "note": "60m sample; V68 stronger phase begins"},
    {"model_tag": "exp_2024_12", "train_start": "2019-01-01", "train_end": "2024-12-31", "test_start": "2025-01-01", "test_end": "2026-04-30", "note": "72m sample; V61 best fixed anchor candidate"},
    {"model_tag": "exp_2025_12", "train_start": "2019-01-01", "train_end": "2025-12-31", "test_start": "2026-01-01", "test_end": "2026-06-30", "note": "84m latest expanding candidate"},
]

# Failure months from V61/V65/V68 plus auto worst months.
MANUAL_FAILURE_MONTHS = [
    "2022-04-01", "2022-08-01", "2023-09-01", "2024-01-02", "2024-03-01", "2025-03-03", "2025-05-06", "2025-11-03", "2026-03-02",
]

RANDOM_SIM_N = 500
RANDOM_SEED = 42

SLIPPAGE_RATE = 0.00246
OPEN_COMMISSION = 0.0003
CLOSE_COMMISSION = 0.0003
CLOSE_TAX = 0.001
BUY_COST_RATE = SLIPPAGE_RATE + OPEN_COMMISSION
SELL_COST_RATE = SLIPPAGE_RATE + CLOSE_COMMISSION + CLOSE_TAX

print("OUT_DIR:", OUT_DIR)
print("model specs:", len(DIAG_MODEL_SPECS))
print("fixed training:", LABEL_BOUNDARY_MODE, "full/direct/fixed", FIXED_ITER)

## 2. V46-family full 特征与分组

分组只用于诊断，不改变训练特征。

In [ ]:
BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio",
    "book_to_price_ratio",
    "earnings_yield",
    "sales_to_price_ratio",
    "cash_earnings_to_price_ratio",
    "earnings_to_price_ratio",
    "roe_ttm",
    "roa_ttm",
    "gross_profit_ttm",
    "operating_profit_to_total_profit",
    "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage",
    "adjusted_profit_to_total_profit",
    "ACCA",
    "growth",
    "net_working_capital",
    "operating_profit_per_share",
    "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share",
    "super_quick_ratio",
    "MLEV",
    "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio",
    "momentum",
    "Rank1M",
    "sharpe_ratio_60",
    "Variance20",
    "liquidity",
    "beta",
    "ATR6",
    "MFI14",
    "DAVOL10",
    "VOL10",
    "VMACD",
    "VOSC",
    "Skewness20",
    "Kurtosis20",
]

HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60",
    "liq_paused_count_20",
    "px_close_to_ma60",
    "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m",
    "ts_Rank1M_rank_chg_1m",
]

FULL_FEATURE_COLS = BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS

FACTOR_GROUPS = {
    "valuation": ["cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio", "cash_earnings_to_price_ratio", "earnings_to_price_ratio"],
    "profit_quality": ["roe_ttm", "roa_ttm", "gross_profit_ttm", "operating_profit_to_total_profit", "adjusted_profit_to_total_profit", "ACCA"],
    "cashflow_balance": ["net_operate_cash_flow_to_total_liability", "net_operating_cash_flow_coverage", "net_operate_cash_flow_per_share", "net_working_capital", "super_quick_ratio"],
    "growth_income": ["growth", "operating_profit_per_share", "total_operating_revenue_per_share"],
    "leverage_risk": ["MLEV", "debt_to_equity_ratio", "debt_to_tangible_equity_ratio", "beta", "Variance20", "ATR6"],
    "momentum_technical": ["momentum", "Rank1M", "sharpe_ratio_60", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC", "Skewness20", "Kurtosis20"],
    "liquidity_price_path": ["liquidity", "liq_money_ratio_20_60", "liq_paused_count_20", "px_close_to_ma60", "px_drawdown_60"],
    "temporal": ["ts_cash_flow_to_price_ratio_rank_mean_3m", "ts_Rank1M_rank_chg_1m"],
}

STYLE_EXPOSURE_COLS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "roe_ttm", "roa_ttm", "growth",
    "momentum", "Rank1M", "sharpe_ratio_60", "Variance20", "beta", "ATR6", "liquidity",
    "liq_money_ratio_20_60", "liq_paused_count_20", "px_close_to_ma60", "px_drawdown_60",
]

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}

feature_group_rows = []
for g, cols in FACTOR_GROUPS.items():
    feature_group_rows.append({"feature_group": g, "feature_count": len(cols), "features": ",".join(cols)})
feature_group_df = pd.DataFrame(feature_group_rows)
feature_group_df.to_csv(OUT_DIR / "v69_feature_groups.csv", index=False)
display_df(feature_group_df)

## 3. Helper 函数

In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


def resolve_data_path():
    if DATA_PATH_OVERRIDE:
        p = Path(DATA_PATH_OVERRIDE)
        if p.exists():
            return p
        raise IOError("DATA_PATH_OVERRIDE not found: %s" % p)
    candidates = [Path(x) for x in DATA_CANDIDATES]
    for p in candidates:
        if p.exists():
            return p
    searched = [str(p.resolve()) for p in candidates]
    raise IOError("training data csv not found. Put train_csi800_factor_v40_data_enhancement*.csv in one of: %s" % searched)


def safe_to_datetime(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce").dt.normalize()
    return out


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def calc_nav(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if len(s) == 0:
        return pd.Series(dtype=float)
    return (1.0 + s).cumprod()


def calc_max_drawdown(ret_series):
    nav = calc_nav(ret_series)
    if len(nav) == 0:
        return np.nan
    return float((nav / nav.cummax() - 1.0).min())


def summarize_returns(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return {"months": 0, "cum_ret": np.nan, "mean_ret": np.nan, "win_rate": np.nan, "max_drawdown": np.nan, "worst_month": np.nan}
    return {
        "months": int(len(s)),
        "cum_ret": float((1.0 + s).prod() - 1.0),
        "mean_ret": float(s.mean()),
        "win_rate": float((s > 0).mean()),
        "max_drawdown": calc_max_drawdown(s),
        "worst_month": float(s.min()),
    }


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            v = corr.iloc[i, j]
            if not pd.isnull(v) and abs(v) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]
    visited = set()
    comps = []

    def dfs(x, comp):
        visited.add(x)
        comp.append(x)
        for y in graph[x]:
            if y not in visited:
                dfs(y, comp)

    for col in feature_cols:
        if col not in visited:
            comp = []
            dfs(col, comp)
            comps.append(comp)
    return comps


def select_features_train_only(train_df, candidate_cols):
    cols = unique_keep_order([c for c in candidate_cols if c in train_df.columns])
    if len(cols) == 0:
        raise ValueError("no candidate feature exists in train data")
    missing = train_df[cols].isnull().sum().to_dict()
    keep = []
    remove = []
    for comp in build_corr_components(train_df, cols, CORR_THRESHOLD):
        if len(comp) == 1:
            keep.append(comp[0])
        else:
            comp = sorted(comp, key=lambda x: (missing[x], x))
            keep.append(comp[0])
            remove.extend(comp[1:])
    return keep, remove


def prepare_xy(df, feature_cols, target_col, fill_values=None):
    d = df.dropna(subset=[target_col]).copy()
    X = d.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    y = d[target_col].astype(float).copy()
    if fill_values is None:
        fill_values = X.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.fillna(fill_values).fillna(0)
    return X, y, fill_values, d.index


def split_diag_valid(train_df):
    months = sorted(pd.to_datetime(train_df[DATE_COL].dropna().unique()))
    if len(months) <= 8:
        return train_df.copy(), train_df.copy()
    n_valid = max(6, int(round(len(months) * 0.20)))
    valid_months = set(months[-min(n_valid, len(months) - 1):])
    fit = train_df[~train_df[DATE_COL].isin(valid_months)].copy()
    valid = train_df[train_df[DATE_COL].isin(valid_months)].copy()
    if fit.empty or valid.empty:
        return train_df.copy(), train_df.copy()
    return fit, valid


def get_stock_board(stock):
    code = str(stock).split(".")[0]
    if code.startswith(("300", "301")):
        return "chinext"
    if code.startswith(("688", "689")):
        return "star"
    return "main"


def board_cap_allows(selected, stock, board_caps):
    board = get_stock_board(stock)
    if board not in board_caps:
        return True
    current = sum(1 for s in selected if get_stock_board(s) == board)
    return current < int(board_caps[board])


def build_board_capped_targets(sorted_stocks, target_num=STOCK_NUM, board_caps=BOARD_CAPS):
    selected = []
    for stock in sorted_stocks:
        if stock in selected:
            continue
        if board_cap_allows(selected, stock, board_caps):
            selected.append(stock)
            if len(selected) >= target_num:
                return selected
    for stock in sorted_stocks:
        if stock not in selected:
            selected.append(stock)
            if len(selected) >= target_num:
                break
    return selected


def zscore_cross_section(s):
    x = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    std = x.std()
    if pd.isnull(std) or std <= 0:
        return x * np.nan
    return (x - x.mean()) / std


def feature_group_of(feature):
    for g, cols in FACTOR_GROUPS.items():
        if feature in cols:
            return g
    return "other"


def list_jaccard(a, b):
    sa, sb = set(a), set(b)
    if not sa and not sb:
        return np.nan
    return len(sa & sb) / float(len(sa | sb))

## 4. 数据加载

In [ ]:
def load_dataset(path):
    df = pd.read_csv(path)
    df = safe_to_datetime(df, [DATE_COL, "feature_date", "next_date"])
    if STOCK_COL not in df.columns:
        for alt in ["code", "security", "order_book_id"]:
            if alt in df.columns:
                df = df.rename(columns={alt: STOCK_COL})
                break
    if TARGET_COL not in df.columns:
        if "raw_return_1m" in df.columns and "benchmark_csi800_1m" in df.columns:
            df[TARGET_COL] = df["raw_return_1m"] - df["benchmark_csi800_1m"]
        else:
            raise ValueError("target column not found: " + TARGET_COL)
    if INDUSTRY_COL not in df.columns:
        df[INDUSTRY_COL] = "UNKNOWN"
    need = [STOCK_COL, DATE_COL, TARGET_COL, INDUSTRY_COL, "feature_date", "next_date"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError("dataset missing columns: " + ",".join(missing))
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=[STOCK_COL, DATE_COL, TARGET_COL]).copy()
    return df


DATA_PATH = resolve_data_path()
df_all = load_dataset(DATA_PATH)

print("DATA_PATH:", DATA_PATH)
print("loaded:", df_all.shape)
print("rebalance_date:", df_all[DATE_COL].min(), "->", df_all[DATE_COL].max())
print("next_date:", df_all["next_date"].min(), "->", df_all["next_date"].max())
print("target:", TARGET_COL)
display_df(df_all[[TARGET_COL]].describe().T)

missing_features = [c for c in FULL_FEATURE_COLS if c not in df_all.columns]
if missing_features:
    print("missing full features:", missing_features)
feature_availability_df = pd.DataFrame([{
    "candidate_features": len(FULL_FEATURE_COLS),
    "available_features": len([c for c in FULL_FEATURE_COLS if c in df_all.columns]),
    "missing_features": ",".join(missing_features),
}])
feature_availability_df.to_csv(OUT_DIR / "v69_feature_availability.csv", index=False)
display_df(feature_availability_df)

## 5. 固定 V46-family 训练与打分

这里训练只是为了诊断模型内部和 score panel；训练口径保持冻结。

In [ ]:
def make_train_df(df_all, spec):
    start = pd.Timestamp(spec["train_start"])
    end = pd.Timestamp(spec["train_end"])
    mask = (df_all[DATE_COL] >= start) & (df_all[DATE_COL] <= end)
    if LABEL_BOUNDARY_MODE == "label_end_safe":
        mask = mask & (df_all["next_date"] <= end)
    elif LABEL_BOUNDARY_MODE != "legacy_rebalance":
        raise ValueError("unknown LABEL_BOUNDARY_MODE: " + str(LABEL_BOUNDARY_MODE))
    return df_all[mask].copy()


def make_test_df(df_all, spec):
    start = pd.Timestamp(spec["test_start"])
    end = pd.Timestamp(spec["test_end"])
    return df_all[(df_all[DATE_COL] >= start) & (df_all[DATE_COL] <= end)].copy()


def train_direct_lgb(train_df, feature_cols):
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED
    X_train, y_train, fill_values, _ = prepare_xy(train_df, feature_cols, TARGET_COL)
    if len(X_train) == 0:
        raise ValueError("empty training matrix")
    model = lgb.train(params, lgb.Dataset(X_train, label=y_train), num_boost_round=max(1, int(FIXED_ITER)))
    pred = np.asarray(model.predict(X_train[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    return {"model": model, "fill_values": fill_values, "train_rows": int(len(X_train)), "train_rank_ic": safe_rank_ic(y_train, pred)}


def score_with_model(df, model, feature_cols, fill_values):
    X = df.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    X = X.fillna(fill_values).fillna(0)
    return np.asarray(model.predict(X[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)


model_bank = {}
model_meta_rows = []
feature_importance_rows = []
group_importance_rows = []
score_panel_parts = []

for spec in progress_iter(DIAG_MODEL_SPECS, total=len(DIAG_MODEL_SPECS), desc="train V46 diagnostics"):
    train_df = make_train_df(df_all, spec)
    test_df = make_test_df(df_all, spec)
    if train_df.empty or test_df.empty:
        print("skip empty", spec["model_tag"], train_df.shape, test_df.shape)
        continue
    diag_fit_df, diag_valid_df = split_diag_valid(train_df)
    feature_cols, removed_cols = select_features_train_only(diag_fit_df, FULL_FEATURE_COLS)
    trained = train_direct_lgb(train_df, feature_cols)
    X_valid, y_valid, _, _ = prepare_xy(diag_valid_df, feature_cols, TARGET_COL, trained["fill_values"])
    valid_pred = np.asarray(trained["model"].predict(X_valid[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    diag_rank_ic = safe_rank_ic(y_valid, valid_pred)
    score_df = test_df.copy()
    score_df["score"] = score_with_model(score_df, trained["model"], feature_cols, trained["fill_values"])
    score_df["model_tag"] = spec["model_tag"]
    score_df["train_start"] = pd.Timestamp(spec["train_start"])
    score_df["train_end"] = pd.Timestamp(spec["train_end"])
    score_df["score_rank_pct"] = score_df.groupby(DATE_COL)["score"].rank(pct=True, method="first")
    score_panel_parts.append(score_df)

    model_bank[spec["model_tag"]] = {"spec": spec, "trained": trained, "feature_cols": feature_cols, "removed_cols": removed_cols, "diag_rank_ic": diag_rank_ic}
    model_meta_rows.append({
        "model_tag": spec["model_tag"],
        "note": spec.get("note", ""),
        "train_start": spec["train_start"],
        "train_end": spec["train_end"],
        "test_start": spec["test_start"],
        "test_end": spec["test_end"],
        "train_rows": int(len(train_df)),
        "train_months": int(train_df[DATE_COL].nunique()),
        "test_rows": int(len(test_df)),
        "test_months": int(test_df[DATE_COL].nunique()),
        "feature_count": int(len(feature_cols)),
        "removed_feature_count": int(len(removed_cols)),
        "train_rank_ic": trained["train_rank_ic"],
        "diag_rank_ic": diag_rank_ic,
        "feature_cols": ",".join(feature_cols),
        "removed_features": ",".join(removed_cols),
    })

    gains = trained["model"].feature_importance(importance_type="gain")
    splits = trained["model"].feature_importance(importance_type="split")
    imp_df = pd.DataFrame({"feature": feature_cols, "importance_gain": gains, "importance_split": splits})
    total_gain = float(imp_df["importance_gain"].sum()) if len(imp_df) else 0.0
    total_split = float(imp_df["importance_split"].sum()) if len(imp_df) else 0.0
    imp_df["importance_gain_pct"] = imp_df["importance_gain"] / total_gain if total_gain > 0 else np.nan
    imp_df["importance_split_pct"] = imp_df["importance_split"] / total_split if total_split > 0 else np.nan
    imp_df["feature_group"] = imp_df["feature"].map(feature_group_of)
    imp_df["model_tag"] = spec["model_tag"]
    imp_df["train_end"] = spec["train_end"]
    feature_importance_rows.extend(imp_df.to_dict("records"))
    grp_gain = imp_df.groupby("feature_group")["importance_gain"].sum().reset_index().rename(columns={"importance_gain": "importance_gain"})
    grp_split = imp_df.groupby("feature_group")["importance_split"].sum().reset_index().rename(columns={"importance_split": "importance_split"})
    grp_count = imp_df.groupby("feature_group")["feature"].size().reset_index().rename(columns={"feature": "feature_count"})
    grp = grp_gain.merge(grp_split, on="feature_group", how="outer").merge(grp_count, on="feature_group", how="outer")
    grp["importance_gain_pct"] = grp["importance_gain"] / grp["importance_gain"].sum() if grp["importance_gain"].sum() > 0 else np.nan
    grp["importance_split_pct"] = grp["importance_split"] / grp["importance_split"].sum() if grp["importance_split"].sum() > 0 else np.nan
    grp["model_tag"] = spec["model_tag"]
    grp["train_end"] = spec["train_end"]
    group_importance_rows.extend(grp.to_dict("records"))
    print("trained", spec["model_tag"], "features", len(feature_cols), "diag_ic", diag_rank_ic)

model_meta_df = pd.DataFrame(model_meta_rows)
feature_importance_df = pd.DataFrame(feature_importance_rows)
group_importance_df = pd.DataFrame(group_importance_rows)
score_panel_df = pd.concat(score_panel_parts, ignore_index=True) if score_panel_parts else pd.DataFrame()

model_meta_df.to_csv(OUT_DIR / "v69_model_meta.csv", index=False)
feature_importance_df.to_csv(OUT_DIR / "v69_feature_importance.csv", index=False)
group_importance_df.to_csv(OUT_DIR / "v69_feature_group_importance.csv", index=False)
score_panel_df.to_csv(OUT_DIR / "v69_score_panel.csv", index=False)

display_df(model_meta_df)
display_df(group_importance_df.sort_values(["model_tag", "importance_gain_pct"], ascending=[True, False]), 20)

## 6. Score bucket 与 top tail 衰减

诊断模型是否只是在 top8 有噪声，还是 top20/top50 整体有单调性。

In [ ]:
def make_score_buckets(score_panel_df):
    rows = []
    if score_panel_df.empty:
        return pd.DataFrame()
    for (model_tag, dt), gdf in progress_iter(score_panel_df.groupby(["model_tag", DATE_COL]), total=score_panel_df.groupby(["model_tag", DATE_COL]).ngroups, desc="score buckets"):
        m = gdf.dropna(subset=["score", TARGET_COL]).copy()
        if m.empty:
            continue
        m = m.sort_values("score", ascending=False).reset_index(drop=True)
        n = len(m)
        bucket_defs = [
            ("top1", 0, 1),
            ("top3", 0, 3),
            ("top8", 0, 8),
            ("rank09_20", 8, 20),
            ("rank21_50", 20, 50),
            ("rank51_100", 50, 100),
            ("bottom50", max(0, n - 50), n),
        ]
        for name, a, b in bucket_defs:
            sub = m.iloc[a:min(b, n)].copy()
            if sub.empty:
                continue
            rows.append({
                "model_tag": model_tag,
                "rebalance_date": dt,
                "bucket": name,
                "count": int(len(sub)),
                "mean_alpha": float(sub[TARGET_COL].mean()),
                "median_alpha": float(sub[TARGET_COL].median()),
                "win_rate": float((sub[TARGET_COL] > 0).mean()),
                "avg_score_rank_pct": float(sub["score_rank_pct"].mean()),
                "targets": ",".join(sub[STOCK_COL].astype(str).tolist()[:20]),
            })
        m["decile"] = pd.qcut(m["score"].rank(method="first"), 10, labels=False, duplicates="drop")
        # qcut labels ascending by rank value; use score quantile label directly for diagnostics.
        for decile, sub in m.groupby("decile"):
            rows.append({
                "model_tag": model_tag,
                "rebalance_date": dt,
                "bucket": "score_decile_%s" % int(decile),
                "count": int(len(sub)),
                "mean_alpha": float(sub[TARGET_COL].mean()),
                "median_alpha": float(sub[TARGET_COL].median()),
                "win_rate": float((sub[TARGET_COL] > 0).mean()),
                "avg_score_rank_pct": float(sub["score_rank_pct"].mean()),
                "targets": "",
            })
    return pd.DataFrame(rows)

bucket_monthly_df = make_score_buckets(score_panel_df)
bucket_summary_rows = []
for (model_tag, bucket), gdf in bucket_monthly_df.groupby(["model_tag", "bucket"]) if len(bucket_monthly_df) else []:
    s = summarize_returns(gdf["mean_alpha"])
    row = {"model_tag": model_tag, "bucket": bucket, "months": int(len(gdf)), "avg_count": float(gdf["count"].mean())}
    for k, v in s.items():
        row[k] = v
    row["avg_monthly_alpha"] = float(gdf["mean_alpha"].mean())
    row["monthly_win_rate"] = float((gdf["mean_alpha"] > 0).mean())
    bucket_summary_rows.append(row)

bucket_summary_df = pd.DataFrame(bucket_summary_rows).sort_values(["model_tag", "bucket"]) if bucket_summary_rows else pd.DataFrame()
bucket_monthly_df.to_csv(OUT_DIR / "v69_score_bucket_monthly.csv", index=False)
bucket_summary_df.to_csv(OUT_DIR / "v69_score_bucket_summary.csv", index=False)

display_df(bucket_summary_df[bucket_summary_df["bucket"].isin(["top1", "top3", "top8", "rank09_20", "rank21_50"])].sort_values(["model_tag", "bucket"]), 30)

## 7. 风格暴露与行业/板块暴露

观察 top8/top20 相对全池在估值、质量、动量、波动、流动性、价格路径上的偏离。

In [ ]:
def selected_targets_from_month(m):
    sorted_stocks = m.sort_values("score", ascending=False)[STOCK_COL].astype(str).tolist()
    return build_board_capped_targets(sorted_stocks, STOCK_NUM, BOARD_CAPS)


def exposure_for_subset(m, selected, prefix):
    out = {}
    sub = m[m[STOCK_COL].astype(str).isin(set(selected))].copy()
    out[prefix + "_count"] = int(len(sub))
    out[prefix + "_alpha_mean"] = float(sub[TARGET_COL].mean()) if len(sub) else np.nan
    for col in STYLE_EXPOSURE_COLS:
        if col not in m.columns:
            continue
        z = zscore_cross_section(m[col])
        sub_z = z.loc[sub.index] if len(sub) else pd.Series(dtype=float)
        out[prefix + "_z_" + col] = float(sub_z.mean()) if len(sub_z) else np.nan
    board_counts = {"main": 0, "chinext": 0, "star": 0}
    for s in selected:
        b = get_stock_board(s)
        board_counts[b] = board_counts.get(b, 0) + 1
    for b, v in board_counts.items():
        out[prefix + "_board_" + b] = int(v)
    if INDUSTRY_COL in m.columns and len(sub):
        vc = sub[INDUSTRY_COL].astype(str).value_counts()
        out[prefix + "_top_industry"] = vc.index[0] if len(vc) else ""
        out[prefix + "_top_industry_count"] = int(vc.iloc[0]) if len(vc) else 0
        out[prefix + "_industry_hhi"] = float(((vc / float(vc.sum())) ** 2).sum()) if vc.sum() > 0 else np.nan
    return out

exposure_rows = []
for (model_tag, dt), gdf in progress_iter(score_panel_df.groupby(["model_tag", DATE_COL]), total=score_panel_df.groupby(["model_tag", DATE_COL]).ngroups if len(score_panel_df) else 0, desc="style exposure"):
    m = gdf.dropna(subset=["score", TARGET_COL]).copy()
    if m.empty:
        continue
    m[STOCK_COL] = m[STOCK_COL].astype(str)
    top8 = selected_targets_from_month(m)
    top20 = m.sort_values("score", ascending=False).head(20)[STOCK_COL].astype(str).tolist()
    row = {"model_tag": model_tag, "rebalance_date": dt, "targets_top8": ",".join(top8), "targets_top20": ",".join(top20)}
    row.update(exposure_for_subset(m, top8, "top8"))
    row.update(exposure_for_subset(m, top20, "top20"))
    exposure_rows.append(row)

exposure_monthly_df = pd.DataFrame(exposure_rows).sort_values(["model_tag", "rebalance_date"]) if exposure_rows else pd.DataFrame()
exposure_summary_rows = []
for model_tag, gdf in exposure_monthly_df.groupby("model_tag") if len(exposure_monthly_df) else []:
    row = {"model_tag": model_tag, "months": int(len(gdf))}
    for col in exposure_monthly_df.columns:
        if col.startswith("top8_z_") or col.startswith("top20_z_") or col.endswith("_industry_hhi"):
            row["avg_" + col] = float(pd.to_numeric(gdf[col], errors="coerce").mean())
    for col in ["top8_board_main", "top8_board_chinext", "top8_board_star", "top20_board_main", "top20_board_chinext", "top20_board_star"]:
        if col in gdf.columns:
            row["avg_" + col] = float(pd.to_numeric(gdf[col], errors="coerce").mean())
    exposure_summary_rows.append(row)

exposure_summary_df = pd.DataFrame(exposure_summary_rows) if exposure_summary_rows else pd.DataFrame()
exposure_monthly_df.to_csv(OUT_DIR / "v69_style_exposure_monthly.csv", index=False)
exposure_summary_df.to_csv(OUT_DIR / "v69_style_exposure_summary.csv", index=False)

display_df(exposure_summary_df, 10)

## 8. 失败月归因

同时看手工失败月和自动最差月份。

In [ ]:
def random_percentile(month_df, selected, n_sim=RANDOM_SIM_N, seed=RANDOM_SEED):
    if len(month_df) == 0 or len(selected) == 0:
        return np.nan
    rng = np.random.RandomState(seed)
    stocks = month_df[STOCK_COL].astype(str).tolist()
    ret_map = dict(zip(month_df[STOCK_COL].astype(str), pd.to_numeric(month_df[TARGET_COL], errors="coerce")))
    selected_ret = np.nanmean([ret_map.get(s, np.nan) for s in selected])
    if pd.isnull(selected_ret):
        return np.nan
    vals = []
    arr = np.arange(len(stocks))
    for _ in range(int(n_sim)):
        perm = rng.permutation(arr)
        ordered = [stocks[i] for i in perm]
        picked = build_board_capped_targets(ordered, STOCK_NUM, BOARD_CAPS)
        vals.append(np.nanmean([ret_map.get(s, np.nan) for s in picked]))
    if len(vals) == 0:
        return np.nan
    return float((np.asarray(vals) <= selected_ret).mean())

failure_dates = set(pd.to_datetime(MANUAL_FAILURE_MONTHS).normalize())
# Add auto worst months by model top8 alpha.
auto_failure_dates = set()
for model_tag, gdf in bucket_monthly_df[bucket_monthly_df["bucket"] == "top8"].groupby("model_tag") if len(bucket_monthly_df) else []:
    tmp = gdf.sort_values("mean_alpha").head(5)
    auto_failure_dates.update(pd.to_datetime(tmp["rebalance_date"]).dt.normalize().tolist())
failure_dates = failure_dates | auto_failure_dates

failure_rows = []
for (model_tag, dt), gdf in progress_iter(score_panel_df.groupby(["model_tag", DATE_COL]), total=score_panel_df.groupby(["model_tag", DATE_COL]).ngroups if len(score_panel_df) else 0, desc="failure attribution"):
    dt_norm = pd.Timestamp(dt).normalize()
    if dt_norm not in failure_dates:
        continue
    m = gdf.dropna(subset=["score", TARGET_COL]).copy()
    if m.empty:
        continue
    m[STOCK_COL] = m[STOCK_COL].astype(str)
    targets = selected_targets_from_month(m)
    top20 = m.sort_values("score", ascending=False).head(20)[STOCK_COL].astype(str).tolist()
    ret_map = dict(zip(m[STOCK_COL], pd.to_numeric(m[TARGET_COL], errors="coerce")))
    target_alpha = np.nanmean([ret_map.get(s, np.nan) for s in targets]) if targets else np.nan
    rank_ic = safe_rank_ic(m["score"], m[TARGET_COL])
    true_top20 = set(m.sort_values(TARGET_COL, ascending=False).head(20)[STOCK_COL].astype(str).tolist())
    true_bottom20 = set(m.sort_values(TARGET_COL, ascending=True).head(20)[STOCK_COL].astype(str).tolist())
    selected_bottom20 = len(set(targets) & true_bottom20)
    row = {
        "model_tag": model_tag,
        "rebalance_date": dt_norm,
        "next_date": pd.Timestamp(m["next_date"].iloc[0]) if "next_date" in m.columns and len(m) else pd.NaT,
        "target_alpha": target_alpha,
        "rank_ic": rank_ic,
        "random_alpha_percentile": random_percentile(m, targets),
        "hit_true_top20": len(set(targets) & true_top20) / float(max(1, len(targets))),
        "selected_bottom20_count": selected_bottom20,
        "targets": ",".join(targets),
        "top20": ",".join(top20),
        "reason_manual_failure_month": bool(dt_norm in set(pd.to_datetime(MANUAL_FAILURE_MONTHS).normalize())),
        "reason_auto_worst_month": bool(dt_norm in auto_failure_dates),
    }
    row.update(exposure_for_subset(m, targets, "target"))
    # Score bucket contrast: top8 vs rank09_20 vs universe.
    ordered = m.sort_values("score", ascending=False).reset_index(drop=True)
    row["top8_alpha"] = float(ordered.head(8)[TARGET_COL].mean()) if len(ordered) else np.nan
    row["rank09_20_alpha"] = float(ordered.iloc[8:20][TARGET_COL].mean()) if len(ordered) > 8 else np.nan
    row["rank21_50_alpha"] = float(ordered.iloc[20:50][TARGET_COL].mean()) if len(ordered) > 20 else np.nan
    row["universe_alpha"] = float(ordered[TARGET_COL].mean()) if len(ordered) else np.nan
    failure_rows.append(row)

failure_attr_df = pd.DataFrame(failure_rows).sort_values(["model_tag", "rebalance_date"]) if failure_rows else pd.DataFrame()
failure_attr_df.to_csv(OUT_DIR / "v69_failure_month_attribution.csv", index=False)
display_df(failure_attr_df[[c for c in ["model_tag", "rebalance_date", "target_alpha", "rank_ic", "random_alpha_percentile", "hit_true_top20", "selected_bottom20_count", "top8_alpha", "rank09_20_alpha", "rank21_50_alpha", "targets"] if c in failure_attr_df.columns]], 40)

## 9. 因子/label 本身诊断

不改变训练，只看每个原始因子在不同年份的 RankIC 和 label 分布，判断问题是否来自样本/标签环境。

In [ ]:
factor_ic_rows = []
for year, ydf in progress_iter(df_all.assign(year=df_all[DATE_COL].dt.year).groupby("year"), total=df_all[DATE_COL].dt.year.nunique(), desc="factor IC by year"):
    for fac in FULL_FEATURE_COLS:
        if fac not in ydf.columns:
            continue
        month_ics = []
        for dt, gdf in ydf.groupby(DATE_COL):
            month_ics.append(safe_rank_ic(gdf[fac], gdf[TARGET_COL]))
        s = pd.Series(month_ics).replace([np.inf, -np.inf], np.nan).dropna()
        factor_ic_rows.append({
            "year": int(year),
            "feature": fac,
            "feature_group": feature_group_of(fac),
            "months": int(len(s)),
            "ic_mean": float(s.mean()) if len(s) else np.nan,
            "ic_median": float(s.median()) if len(s) else np.nan,
            "ic_ir": float(s.mean() / s.std()) if len(s) > 1 and s.std() > 0 else np.nan,
            "ic_positive_rate": float((s > 0).mean()) if len(s) else np.nan,
        })

factor_ic_yearly_df = pd.DataFrame(factor_ic_rows)
if len(factor_ic_yearly_df):
    gkeys = ["year", "feature_group"]
    group_ic_count = factor_ic_yearly_df.groupby(gkeys)["feature"].size().reset_index().rename(columns={"feature": "feature_count"})
    group_ic_avg = factor_ic_yearly_df.groupby(gkeys)["ic_mean"].mean().reset_index().rename(columns={"ic_mean": "avg_ic_mean"})
    group_ic_median = factor_ic_yearly_df.groupby(gkeys)["ic_mean"].median().reset_index().rename(columns={"ic_mean": "median_ic_mean"})
    group_ic_pos = factor_ic_yearly_df.groupby(gkeys)["ic_positive_rate"].mean().reset_index().rename(columns={"ic_positive_rate": "avg_positive_rate"})
    group_ic_yearly_df = group_ic_count.merge(group_ic_avg, on=gkeys, how="outer").merge(group_ic_median, on=gkeys, how="outer").merge(group_ic_pos, on=gkeys, how="outer")
else:
    group_ic_yearly_df = pd.DataFrame()

label_rows = []
for year, ydf in df_all.assign(year=df_all[DATE_COL].dt.year).groupby("year"):
    s = pd.to_numeric(ydf[TARGET_COL], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    month_disp = []
    for dt, gdf in ydf.groupby(DATE_COL):
        x = pd.to_numeric(gdf[TARGET_COL], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        if len(x):
            month_disp.append(x.quantile(0.9) - x.quantile(0.1))
    label_rows.append({
        "year": int(year),
        "rows": int(len(s)),
        "mean_alpha": float(s.mean()) if len(s) else np.nan,
        "median_alpha": float(s.median()) if len(s) else np.nan,
        "std_alpha": float(s.std()) if len(s) else np.nan,
        "p10_alpha": float(s.quantile(0.10)) if len(s) else np.nan,
        "p90_alpha": float(s.quantile(0.90)) if len(s) else np.nan,
        "cross_section_dispersion_p90_p10_avg": float(pd.Series(month_disp).mean()) if len(month_disp) else np.nan,
        "positive_rate": float((s > 0).mean()) if len(s) else np.nan,
    })

label_yearly_df = pd.DataFrame(label_rows)
factor_ic_yearly_df.to_csv(OUT_DIR / "v69_factor_ic_yearly.csv", index=False)
group_ic_yearly_df.to_csv(OUT_DIR / "v69_factor_group_ic_yearly.csv", index=False)
label_yearly_df.to_csv(OUT_DIR / "v69_label_yearly_diagnostics.csv", index=False)

display_df(group_ic_yearly_df.sort_values(["year", "avg_ic_mean"], ascending=[True, False]), 30)
display_df(label_yearly_df, 20)

## 10. 汇总提示

这张表是诊断索引，不是交易建议。

In [ ]:
summary_rows = []
for model_tag, meta in model_meta_df.set_index("model_tag").iterrows() if len(model_meta_df) else []:
    b = bucket_summary_df[(bucket_summary_df["model_tag"] == model_tag) & (bucket_summary_df["bucket"].isin(["top8", "rank09_20", "rank21_50"]))]
    exp = exposure_summary_df[exposure_summary_df["model_tag"] == model_tag] if len(exposure_summary_df) else pd.DataFrame()
    fail = failure_attr_df[failure_attr_df["model_tag"] == model_tag] if len(failure_attr_df) else pd.DataFrame()
    top8 = b[b["bucket"] == "top8"]
    r0920 = b[b["bucket"] == "rank09_20"]
    r2150 = b[b["bucket"] == "rank21_50"]
    row = {
        "model_tag": model_tag,
        "train_end": meta.get("train_end"),
        "test_start": meta.get("test_start"),
        "test_end": meta.get("test_end"),
        "train_months": meta.get("train_months"),
        "feature_count": meta.get("feature_count"),
        "train_rank_ic": meta.get("train_rank_ic"),
        "diag_rank_ic": meta.get("diag_rank_ic"),
        "top8_cum_alpha": float(top8["cum_ret"].iloc[0]) if len(top8) else np.nan,
        "top8_monthly_win_rate": float(top8["monthly_win_rate"].iloc[0]) if len(top8) else np.nan,
        "rank09_20_cum_alpha": float(r0920["cum_ret"].iloc[0]) if len(r0920) else np.nan,
        "rank21_50_cum_alpha": float(r2150["cum_ret"].iloc[0]) if len(r2150) else np.nan,
        "failure_month_count": int(len(fail)),
        "failure_avg_random_pct": float(fail["random_alpha_percentile"].mean()) if len(fail) and "random_alpha_percentile" in fail.columns else np.nan,
        "failure_avg_target_alpha": float(fail["target_alpha"].mean()) if len(fail) and "target_alpha" in fail.columns else np.nan,
    }
    if len(exp):
        for col in ["avg_top8_z_beta", "avg_top8_z_Variance20", "avg_top8_z_momentum", "avg_top8_z_Rank1M", "avg_top8_z_cash_flow_to_price_ratio", "avg_top8_z_px_drawdown_60", "avg_top8_board_chinext", "avg_top8_board_star", "avg_top8_industry_hhi"]:
            if col in exp.columns:
                row[col] = float(exp[col].iloc[0])
    if row["top8_cum_alpha"] > 0 and row["rank09_20_cum_alpha"] > 0:
        row["diagnostic_hint"] = "signal_extends_beyond_top8"
    elif row["top8_cum_alpha"] > 0 and (pd.isnull(row["rank09_20_cum_alpha"]) or row["rank09_20_cum_alpha"] <= 0):
        row["diagnostic_hint"] = "top_tail_sensitive"
    else:
        row["diagnostic_hint"] = "weak_test_window"
    summary_rows.append(row)

v69_summary_df = pd.DataFrame(summary_rows)
v69_summary_df.to_csv(OUT_DIR / "v69_diagnostic_summary.csv", index=False)

print("saved outputs:")
for fp in sorted(OUT_DIR.glob("v69_*.csv")):
    print("-", fp)

display_df(v69_summary_df, 20)